# Optional Project - Colab Part4 Training and Generation

Runs Part 4: continue training the best model, generate unconditional and prefix-conditioned SVG samples, render them, evaluate validity metrics, and sync checkpoints/results to Google Drive.

In [ ]:
# ===== User config =====
REPO_URL = "https://github.com/Peng-y-x/optionalproject.git"
REPO_DIR = "/content/optionalproject"
REPO_BRANCH = "run"
DATA_CONFIG = "configs/data.yaml"
PART4_CONFIG = "configs/part4_best.yaml"
GEN_CONFIG = "configs/part4_generation.yaml"
BEST_LR_JSON = "outputs/part3_mup_lr_sweep/best_lr.json"
DRIVE_BEST_LR_JSON = "/content/drive/MyDrive/svg-scaling/part3_mup_lr_sweep/best_lr.json"
DRIVE_ROOT = "/content/drive/MyDrive/svg-scaling"
MAX_EPOCHS = 3
# Optional cap for limited Colab sessions; set to 0 for full epochs.
MAX_TRAIN_TOKENS_PER_EPOCH = 0


In [ ]:
# 1) Clone repo and checkout branch
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    %cd $REPO_DIR
    !git fetch origin {REPO_BRANCH}
    !git checkout {REPO_BRANCH}
    !git pull --ff-only origin {REPO_BRANCH}


In [ ]:
# 2) Mount Google Drive for resumable Part 4 checkpoints and final artifacts
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p {DRIVE_ROOT}/part4_best {DRIVE_ROOT}/part4_samples {DRIVE_ROOT}/part4_eval {DRIVE_ROOT}/tokenizer


In [ ]:
# 3) Install system + Python dependencies
!apt-get update -y
!apt-get install -y libcairo2 libcairo2-dev libffi-dev
!python -m pip install -q -r {REPO_DIR}/requirements.txt


In [ ]:
# 4) HF auth from Colab Keys (key name must be HF_TOKEN)
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
print('has_hf_token:', bool(os.getenv('HF_TOKEN')))


In [ ]:
# 5) GPU sanity check
import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))


In [ ]:
# 6) Prepare tokenizer artifacts for generation/evaluation.
# Training reads token IDs from HF dataset Zala0429/svg-scaling-v2-tokenized, as in Parts 2-3.
# Generation needs the matching BPE tokenizer; restore from Drive if available, otherwise rebuild from HF clean raw data.
%cd $REPO_DIR
from pathlib import Path
import json, os, shutil, subprocess, yaml
from datasets import load_dataset

cfg = yaml.safe_load(Path(DATA_CONFIG).read_text(encoding='utf-8'))
raw_repo = cfg['hf_push']['repo_id']
raw_dir = Path(cfg['output']['dir'])
tok_dir = Path(cfg['tokenization']['output_dir'])
drive_tok_dir = Path(DRIVE_ROOT) / 'tokenizer'

if (drive_tok_dir / 'tokenizer.json').exists() and not (tok_dir / 'tokenizer.json').exists():
    tok_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(drive_tok_dir, tok_dir, dirs_exist_ok=True)
    print('Restored tokenizer from Drive:', drive_tok_dir)

if not (tok_dir / 'tokenizer.json').exists():
    print('Downloading clean raw splits from HF:', raw_repo)
    raw_dir.mkdir(parents=True, exist_ok=True)
    ds = load_dataset(raw_repo)
    for split, fname in [('train','train.jsonl'), ('validation','validation.jsonl'), ('test','test.jsonl')]:
        with open(raw_dir / fname, 'w', encoding='utf-8') as f:
            for row in ds[split]:
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
    subprocess.run(['python', 'scripts/run_tokenizer.py', '--config', DATA_CONFIG], check=True)
    shutil.copytree(tok_dir, drive_tok_dir, dirs_exist_ok=True)
    print('Tokenizer rebuilt and synced to Drive:', drive_tok_dir)

print('tokenizer:', tok_dir / 'tokenizer.json')


In [ ]:
# 7) Resolve Part 3 best LR and initialization checkpoint.
# The Part 4 run writes to its own Drive directory and resumes from there after disconnects.
%cd $REPO_DIR
from pathlib import Path
import shutil

if not Path(BEST_LR_JSON).exists() and Path(DRIVE_BEST_LR_JSON).exists():
    Path(BEST_LR_JSON).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_BEST_LR_JSON, BEST_LR_JSON)

init_candidates = [
    Path(DRIVE_ROOT) / 'part3_mup/xl_mup/checkpoints/best.pt',
    Path(DRIVE_ROOT) / 'part3_mup/xl_mup/checkpoints/latest.pt',
    Path(DRIVE_ROOT) / 'part2_v2/xl/checkpoints/best.pt',
    Path(DRIVE_ROOT) / 'part2_v2/xl/checkpoints/latest.pt',
]
INIT_FROM = next((str(p) for p in init_candidates if p.exists()), '')
print('best_lr_json exists:', Path(BEST_LR_JSON).exists())
print('init_from:', INIT_FROM or '(none; train from scratch)')


In [ ]:
# 8) Continue training best model for Part 4.
# If Colab disconnects, rerun this cell; it resumes from /content/drive/MyDrive/svg-scaling/part4_best/latest.pt.
%cd $REPO_DIR
import subprocess
from pathlib import Path

cmd = ['python', 'scripts/run_part4_train.py', '--config', PART4_CONFIG, '--parameterization', 'mup', '--max-epochs', str(MAX_EPOCHS)]
if Path(BEST_LR_JSON).exists():
    cmd += ['--best-lr-json', BEST_LR_JSON]
else:
    cmd += ['--learning-rate', '0.0006']
if INIT_FROM:
    cmd += ['--init-from', INIT_FROM]
if MAX_TRAIN_TOKENS_PER_EPOCH and MAX_TRAIN_TOKENS_PER_EPOCH > 0:
    cmd += ['--max-train-tokens-per-epoch', str(MAX_TRAIN_TOKENS_PER_EPOCH)]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 9) Inspect Part 4 training output
%cd $REPO_DIR
from pathlib import Path
import json
run_dir = Path('outputs/part4_best/best_mup_xl')
if not run_dir.exists():
    run_dir = Path(DRIVE_ROOT) / 'part4_best/best_mup_xl'
print('run_dir:', run_dir)
for p in [run_dir / 'summary.json', run_dir / 'final_metrics.json']:
    print('\n', p, p.exists())
    if p.exists():
        print(json.dumps(json.loads(p.read_text()), indent=2)[:2000])


In [ ]:
# 10) Generate unconditional and prefix-conditioned SVG samples, render PNGs, and sync to Drive.
%cd $REPO_DIR
from pathlib import Path
import subprocess

ckpt_candidates = [
    Path('outputs/part4_best/best_mup_xl/checkpoints/best.pt'),
    Path('outputs/part4_best/best_mup_xl/checkpoints/latest.pt'),
    Path(DRIVE_ROOT) / 'part4_best/best_mup_xl/checkpoints/best.pt',
    Path(DRIVE_ROOT) / 'part4_best/best_mup_xl/checkpoints/latest.pt',
]
PART4_CKPT = next(str(p) for p in ckpt_candidates if p.exists())
TOKENIZER_PATH = 'data/processed/v1-clean-rawsplit/tokenizer/tokenizer.json'
cmd = [
    'python', 'scripts/run_generate.py',
    '--config', GEN_CONFIG,
    '--checkpoint-path', PART4_CKPT,
    '--tokenizer-path', TOKENIZER_PATH,
    '--output-dir', 'outputs/part4_samples',
    '--drive-output-dir', f'{DRIVE_ROOT}/part4_samples',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 11) Evaluate Part 4: test perplexity + XML/render/structural validity for generated samples.
%cd $REPO_DIR
import subprocess
cmd = [
    'python', 'scripts/run_eval.py',
    '--train-config', PART4_CONFIG,
    '--checkpoint-path', PART4_CKPT,
    '--samples-jsonl', 'outputs/part4_samples/samples.jsonl',
    '--output-dir', 'outputs/part4_eval',
    '--drive-output-dir', f'{DRIVE_ROOT}/part4_eval',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# 12) Inspect final Part 4 artifacts
%cd $REPO_DIR
from pathlib import Path
import json
for p in [
    Path('outputs/part4_samples/generation_summary.json'),
    Path('outputs/part4_eval/part4_metrics.json'),
    Path('outputs/part4_samples/generated_grid.png'),
]:
    print('\n', p, p.exists())
    if p.suffix == '.json' and p.exists():
        print(json.dumps(json.loads(p.read_text()), indent=2)[:2000])
print('Drive results:', f'{DRIVE_ROOT}/part4_samples', f'{DRIVE_ROOT}/part4_eval')
